In [ ]:
from pathlib import Path
import pynapple as nap
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np

session_name = 'LOW_10Hz'

def plot_opto_responses(session_name, time_bs=0.002):
    # load stuff in 
    deriv_folder = Path('/Volumes/cmvm/sbms/groups/CDBS_SIDB_storage/NolanLab/ActiveProjects/Wolf/OPTO_PILOT_MECL5A_CHRIMSONR/derivatives') / session_name
    spikes = nap.load_file(deriv_folder / f'{session_name}_spikes_curated.npz')
    start_stim = nap.load_file(deriv_folder / f'{session_name}_start.npz')
    stop_stim = nap.load_file(deriv_folder / f'{session_name}_stop.npz')

    # Now use spikes_z in perievent calculation
    perievent = nap.compute_perievent(
        timestamps=spikes,
        tref=start_stim,
        minmax=(-0.01, 0.09),
        time_unit="s"
    )

    import math

    n_units = len(spikes.index)
    cells_per_row = 5
    n_rows = math.ceil(n_units / cells_per_row)

    fig, axes = plt.subplots(n_rows, cells_per_row * 2, figsize=(3 * cells_per_row * 2, 2.5 * n_rows), sharex=True, sharey=False)
    axes = np.array(axes).reshape(n_rows, cells_per_row * 2)

    for idx, unit_id in enumerate(spikes.index):
        row = idx // cells_per_row
        col = (idx % cells_per_row) * 2
        # Firing rate
        rate = np.mean(perievent[unit_id].count(time_bs), 1) / time_bs
        avg_rate = np.mean(rate)
        ax0 = axes[row, col]
        ax0.plot(rate, linewidth=1, color="black")
        ax0.set_xlim(-0.01, 0.09)
        ax0.set_ylabel("Rate (spikes/sec)")
        ax0.axvspan(-0.005, 0, ymin=0, ymax=1, color='grey', alpha=0.3)
        ax0.axvline(0.0)
        ax0.set_title(f"Unit {unit_id}")
        # Raster
        tsd = perievent[unit_id].to_tsd()
        ax1 = axes[row, col + 1]
        # Inverse scaling: alpha = 0.3 / avg_rate
        ax1.plot(tsd, "o", markersize=2, color="black", alpha=0.1, mew=4)
        ax1.set_xlabel("Time from stim (s)")
        ax1.set_ylabel("Stimulus")
        ax1.set_xlim(-0.01, 0.09)
        ax1.axvspan(-0.005, 0, ymin=0, ymax=1, color='grey', alpha=0.3)
        ax1.axvline(0.0)

    # Hide unused axes
    for idx in range(n_units, n_rows * cells_per_row):
        row = idx // cells_per_row
        col = (idx % cells_per_row) * 2
        axes[row, col].axis('off')
        axes[row, col + 1].axis('off')

    plt.tight_layout()
    plt.show()

In [20]:
for session_name in ['HIGH_10Hz', 'MED_10Hz', 'LOW_10Hz']:
    print(f'Processing session: {session_name}')
    plot_opto_responses(session_name)

Processing session: HIGH_10Hz


NameError: name 'spikes_z' is not defined